<a href="https://colab.research.google.com/github/rchirutkar/AgenticAI/blob/main/Week_6_Day_2_Learner_RAG_SQLite_Memory_HRBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Learner Notebook — Day 2  
## RAG + SQLite Memory: Complete HR Bot Flow

**Session length:** 2.5 hours  
**Audience:** Python developers  
**Project goal:** Combine Day 1 memory with a real retrieval pipeline.

Today you will build:

1. A SQLite-backed vector-like policy store.
2. Chunk → embed → store → retrieve flow.
3. A complete HR Bot prompt package using:
   - short-term summary,
   - last 10 episodic messages,
   - employee tools,
   - retrieved policy chunks.
4. The missing employee-ID flow versus remembered employee-ID flow.

## Colab-first and no-paid-API design

The core lab uses SQLite, deterministic summarisation, and local retrieval so every learner can complete it without purchasing API credits. An optional production extension may use Groq or OpenRouter for final response generation, with keys stored in Colab Secrets.

## What Is Different from Day 1?

Day 1 focused on memory:

```text
employee ID + last messages + summary
```

Day 2 adds long-term policy knowledge through retrieval:

```text
policy document → chunks → embeddings → SQLite policy store → top chunks → prompt
```

In production, the vector store could be Pinecone, Weaviate, Qdrant, pgvector, or another system. In this notebook, SQLite stores the chunks and vectors locally so everyone can run the full flow.

## Day 2 Agenda — 2.5 Hours

| Time | Segment | Output |
|---:|---|---|
| 0:00–0:20 | Recap Day 1 memory | Learners explain last-10 + summary |
| 0:20–0:45 | RAG architecture | Learners trace chunk → retrieve → answer |
| 0:45–1:15 | Build SQLite policy store | Store chunks and vectors |
| 1:15–1:25 | Break | Reset |
| 1:25–2:05 | Integrate HR Bot | Memory + employee tools + policy retrieval |
| 2:05–2:25 | Test scenarios | Missing ID vs remembered ID |
| 2:25–2:30 | Wrap-up | Submit extension ideas |

# 1. Setup

Run these cells even if you completed Day 1. They recreate the database objects if needed.

In [ ]:
%pip install -q pandas openpyxl

from pathlib import Path
import sqlite3
import pandas as pd
import json
import re
from datetime import datetime


def locate_file(filename: str) -> Path:
    """Find a support file in common local and Google Colab locations."""
    candidates = [Path(filename), Path('/content') / filename]
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidates.append(parent / filename)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. In Colab, upload it through the Files panel or run files.upload()."
    )


EMPLOYEE_FILE = locate_file("employees.xlsx")
LEAVE_FILE = locate_file("leave_balance.xlsx")
DATA_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
DB_FILE = DATA_DIR / "hr_bot_memory.db"

print("Employee file:", EMPLOYEE_FILE)
print("Leave file:", LEAVE_FILE)
print("Using database:", DB_FILE.resolve())

Employee file: /content/employees.xlsx
Leave file: /content/leave_balance.xlsx
Using database: /content/hr_bot_memory.db


In [ ]:
def load_source_files_to_sqlite(db_file=DB_FILE):
    """Load the supplied HR sample spreadsheets into real SQLite tables."""
    employees = pd.read_excel(EMPLOYEE_FILE)

    leave_raw = pd.read_excel(LEAVE_FILE, header=2)
    leave = leave_raw[leave_raw["Employee ID"].astype(str).str.startswith("EMP-", na=False)].copy()

    leave = leave.rename(columns={
        "Entitled": "annual_entitled",
        "Taken": "annual_taken",
        "Balance": "annual_balance",
        "Entitled.1": "sick_entitled",
        "Taken.1": "sick_taken",
        "Balance.1": "sick_balance",
        "Entitled.2": "casual_entitled",
        "Taken.2": "casual_taken",
        "Balance.2": "casual_balance",
        "Comp Off Balance": "comp_off_balance",
        "Leave Year": "leave_year",
        "Status": "status",
    })

    with sqlite3.connect(db_file) as conn:
        employees.to_sql("employees", conn, if_exists="replace", index=False)
        leave.to_sql("leave_balances", conn, if_exists="replace", index=False)

    return employees.shape, leave.shape

employee_shape, leave_shape = load_source_files_to_sqlite()
employee_shape, leave_shape

((20, 9), (20, 14))

In [ ]:
def query_one(sql: str, params=()):
    """Return one SQLite row as a Python dict."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(sql, params).fetchone()
    return dict(row) if row else None

def get_employee_details(emp_id: str) -> dict:
    row = query_one(
        """
        SELECT "Employee ID" AS employee_id,
               Name AS name,
               Designation AS designation,
               "Reporting Manager" AS reporting_manager
        FROM employees
        WHERE "Employee ID" = ?
        """,
        (emp_id,),
    )
    return row or {"error": f"Employee {emp_id} not found"}

def get_leave_balance(emp_id: str) -> dict:
    row = query_one(
        """
        SELECT "Employee ID" AS employee_id,
               Name AS name,
               annual_balance,
               sick_balance,
               casual_balance,
               comp_off_balance,
               leave_year,
               status
        FROM leave_balances
        WHERE "Employee ID" = ?
        """,
        (emp_id,),
    )
    return row or {"error": f"No leave record for {emp_id}"}

print(get_employee_details("EMP-1001"))
print(get_leave_balance("EMP-1001"))

{'employee_id': 'EMP-1001', 'name': 'Priya Sharma', 'designation': 'Software Engineer', 'reporting_manager': 'Ravi Kumar'}
{'employee_id': 'EMP-1001', 'name': 'Priya Sharma', 'annual_balance': 13, 'sick_balance': 9, 'casual_balance': 4, 'comp_off_balance': 3, 'leave_year': '2025-26', 'status': '✔ Healthy'}


In [ ]:
MAX_EPISODIC_MESSAGES = 10

def init_memory_tables(db_file=DB_FILE):
    """Create real persistent memory tables in SQLite."""
    with sqlite3.connect(db_file) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS chat_messages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                session_id TEXT NOT NULL,
                role TEXT NOT NULL CHECK(role IN ('user', 'assistant', 'system')),
                content TEXT NOT NULL,
                created_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.execute("""
            CREATE TABLE IF NOT EXISTS session_memory (
                session_id TEXT PRIMARY KEY,
                summary TEXT NOT NULL DEFAULT '',
                current_employee_id TEXT,
                updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.execute("CREATE INDEX IF NOT EXISTS idx_chat_session_id ON chat_messages(session_id, id)")
        conn.commit()

init_memory_tables()
print("Memory tables ready.")

Memory tables ready.


In [ ]:
def extract_emp_id(text: str):
    """Extract employee IDs such as EMP-1001 from free text."""
    match = re.search(r"\bEMP-\d+\b", text.upper())
    return match.group(0) if match else None

def get_recent_messages(session_id: str, limit: int = MAX_EPISODIC_MESSAGES):
    """Fetch only the recent episodic memory window."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT id, role, content, created_at
            FROM chat_messages
            WHERE session_id = ?
            ORDER BY id DESC
            LIMIT ?
            """,
            (session_id, limit),
        ).fetchall()
    return [dict(row) for row in reversed(rows)]

def get_session_memory(session_id: str):
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        row = conn.execute(
            "SELECT session_id, summary, current_employee_id, updated_at FROM session_memory WHERE session_id = ?",
            (session_id,),
        ).fetchone()
    if row:
        return dict(row)
    return {"session_id": session_id, "summary": "", "current_employee_id": None, "updated_at": None}

def upsert_session_memory(session_id: str, summary=None, current_employee_id=None):
    """Update durable short-term memory. Existing values are preserved unless a new value is supplied."""
    existing = get_session_memory(session_id)
    final_summary = existing["summary"] if summary is None else summary
    final_emp_id = existing["current_employee_id"] if current_employee_id is None else current_employee_id

    with sqlite3.connect(DB_FILE) as conn:
        conn.execute(
            """
            INSERT INTO session_memory(session_id, summary, current_employee_id, updated_at)
            VALUES (?, ?, ?, CURRENT_TIMESTAMP)
            ON CONFLICT(session_id) DO UPDATE SET
                summary = excluded.summary,
                current_employee_id = excluded.current_employee_id,
                updated_at = CURRENT_TIMESTAMP
            """,
            (session_id, final_summary, final_emp_id),
        )
        conn.commit()

def compress_messages(existing_summary: str, older_messages: list[dict]) -> str:
    """
    Compress older chat messages into durable short-term memory.

    In production, this can call an LLM summarizer. For the workshop, this is a deterministic
    compression step so every learner can run it without an API key.
    """
    if not older_messages:
        return existing_summary or ""

    employee_ids = sorted({
        emp_id
        for msg in older_messages
        for emp_id in [extract_emp_id(msg["content"])]
        if emp_id
    })

    topic_keywords = []
    text = " ".join(msg["content"].lower() for msg in older_messages)
    for keyword in ["leave", "manager", "policy", "sick", "casual", "annual", "approval"]:
        if keyword in text:
            topic_keywords.append(keyword)

    compressed_turns = " | ".join(
        f'{msg["role"]}: {msg["content"][:90].replace(chr(10), " ")}'
        for msg in older_messages[-6:]
    )

    new_summary_piece = (
        f"Compressed {len(older_messages)} older messages. "
        f"Employee IDs mentioned: {employee_ids or 'none'}. "
        f"Topics: {topic_keywords or 'general HR chat'}. "
        f"Recent older context: {compressed_turns}"
    )

    if existing_summary:
        combined = existing_summary.strip() + "\n" + new_summary_piece
    else:
        combined = new_summary_piece

    # Keep the durable summary compact enough to place into prompts.
    return combined[-1800:]

def compact_episodic_memory(session_id: str, keep_last: int = MAX_EPISODIC_MESSAGES):
    """
    Keep only the last N chat messages in episodic memory.
    Move everything before that into short-term summary memory.
    """
    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute(
            """
            SELECT id, role, content, created_at
            FROM chat_messages
            WHERE session_id = ?
            ORDER BY id ASC
            """,
            (session_id,),
        ).fetchall()

    rows = [dict(row) for row in rows]
    if len(rows) <= keep_last:
        return {"compacted": 0, "kept": len(rows)}

    older = rows[:-keep_last]
    keep = rows[-keep_last:]

    existing = get_session_memory(session_id)
    updated_summary = compress_messages(existing["summary"], older)
    upsert_session_memory(session_id, summary=updated_summary)

    older_ids = [row["id"] for row in older]
    placeholders = ",".join("?" for _ in older_ids)
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute(f"DELETE FROM chat_messages WHERE id IN ({placeholders})", older_ids)
        conn.commit()

    return {"compacted": len(older), "kept": len(keep)}

def save_message(session_id: str, role: str, content: str):
    """Save a chat message, then enforce the 'last 10 messages only' episodic window."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute(
            "INSERT INTO chat_messages(session_id, role, content) VALUES (?, ?, ?)",
            (session_id, role, content),
        )
        conn.commit()
    return compact_episodic_memory(session_id)

def reset_session(session_id: str):
    """Convenience function for repeated demos."""
    with sqlite3.connect(DB_FILE) as conn:
        conn.execute("DELETE FROM chat_messages WHERE session_id = ?", (session_id,))
        conn.execute("DELETE FROM session_memory WHERE session_id = ?", (session_id,))
        conn.commit()

print("Memory functions ready.")

Memory functions ready.


In [ ]:
def resolve_employee_id(session_id: str, user_message: str):
    """
    Resolve employee ID in this order:
    1. Current user message.
    2. Durable short-term memory for the present chat session.
    3. Last 10 episodic chat messages.
    4. Missing → bot should ask the user.
    """
    emp_id = extract_emp_id(user_message)
    if emp_id:
        upsert_session_memory(session_id, current_employee_id=emp_id)
        return emp_id, "current_message"

    session_memory = get_session_memory(session_id)
    if session_memory["current_employee_id"]:
        return session_memory["current_employee_id"], "short_term_session_memory"

    for msg in reversed(get_recent_messages(session_id)):
        emp_id = extract_emp_id(msg["content"])
        if emp_id:
            upsert_session_memory(session_id, current_employee_id=emp_id)
            return emp_id, "episodic_recent_messages"

    return None, "missing"

def answer_leave_balance(emp_id: str) -> str:
    details = get_leave_balance(emp_id)
    if "error" in details:
        return details["error"]
    return (
        f'{details["name"]} ({emp_id}) has '
        f'{int(details["annual_balance"])} annual leave days, '
        f'{int(details["sick_balance"])} sick leave days, '
        f'{int(details["casual_balance"])} casual leave days, and '
        f'{int(details["comp_off_balance"])} comp-off days available '
        f'for {details["leave_year"]}.'
    )

def answer_manager(emp_id: str) -> str:
    details = get_employee_details(emp_id)
    if "error" in details:
        return details["error"]
    return f'{details["name"]} ({emp_id}) reports to {details["reporting_manager"]}.'

def hr_bot(session_id: str, user_message: str) -> str:
    """
    A small but real tool-orchestration bot:
    - persists every turn to SQLite,
    - resolves employee ID from current/session/recent memory,
    - calls SQLite tools for facts,
    - asks for missing employee ID when needed.
    """
    save_message(session_id, "user", user_message)

    emp_id, source = resolve_employee_id(session_id, user_message)
    lower = user_message.lower()

    needs_employee_record = any(word in lower for word in ["my", "leave", "manager", "balance", "days"])

    if not emp_id and needs_employee_record:
        response = (
            "I can help with that, but I do not know which employee record to check in this chat session. "
            "Please share your employee ID, for example: EMP-1001."
        )
    elif "leave" in lower or "balance" in lower or "days" in lower:
        response = answer_leave_balance(emp_id) + f" [Employee ID source: {source}]"
    elif "manager" in lower or "reporting" in lower:
        response = answer_manager(emp_id) + f" [Employee ID source: {source}]"
    elif emp_id:
        response = f"Thanks. I will use {emp_id} for this chat session. [Employee ID source: {source}]"
    else:
        response = "I can help with leave balance, reporting manager, and HR policy questions."

    save_message(session_id, "assistant", response)
    return response

# 2. Build a Local Policy Retrieval Store

This is a real retrieval pipeline:

1. Split policy text into chunks.
2. Create numeric vectors for each chunk.
3. Store chunks and vectors in SQLite.
4. Convert a user question into a vector.
5. Retrieve the most similar chunks.

The embedding function is intentionally lightweight so the class can run without external APIs.

In [ ]:
import math
import hashlib
from collections import Counter

def tokenize(text: str):
    return re.findall(r"[a-zA-Z][a-zA-Z0-9_-]+", text.lower())

def embed_text(text: str, dims: int = 128):
    """
    A lightweight local embedding for teaching the vector-store concept.
    It hashes tokens into a fixed-length numeric vector.

    Production swap:
    - replace this with a local sentence-transformer or approved embedding service,
    - store vectors in Pinecone, Weaviate, Qdrant, or pgvector.
    """
    vector = [0.0] * dims
    counts = Counter(tokenize(text))
    for token, count in counts.items():
        idx = int(hashlib.md5(token.encode("utf-8")).hexdigest(), 16) % dims
        vector[idx] += float(count)
    norm = math.sqrt(sum(x * x for x in vector)) or 1.0
    return [x / norm for x in vector]

def cosine_similarity(a, b):
    return sum(x * y for x, y in zip(a, b))

def chunk_text(text: str, chunk_size_words: int = 90, overlap_words: int = 20):
    words = text.split()
    chunks = []
    start = 0
    chunk_id = 1

    while start < len(words):
        end = min(start + chunk_size_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append({"chunk_id": f"policy_chunk_{chunk_id:03d}", "text": chunk})
        if end == len(words):
            break
        start = end - overlap_words
        chunk_id += 1

    return chunks

In [ ]:
SAMPLE_HR_POLICY = """
Annual Leave Policy:
Employees are entitled to annual leave according to their employment contract and leave register.
Annual leave should normally be requested at least 5 working days in advance.
Requests longer than 3 consecutive working days require reporting manager approval.
Employees should not plan annual leave that exceeds their available annual leave balance.

Sick Leave Policy:
Employees may use sick leave for illness, medical appointments, or recovery.
For sick leave longer than 2 consecutive working days, the employee may be asked to submit medical documentation.
Sick leave is not normally encashed.

Casual Leave Policy:
Casual leave can be used for urgent personal work or short unplanned absences.
Casual leave should be requested as early as possible.
Manager approval is required before the absence unless it is an emergency.

Compensatory Off Policy:
Comp-off may be granted when an employee works on a weekend or company holiday with prior approval.
Comp-off must usually be used within the same leave year unless HR approves an exception.

Approval Workflow:
The employee submits a leave request.
The reporting manager reviews business continuity and leave balance.
HR may review exceptions, long leave, policy conflicts, or insufficient balance cases.
"""

def init_policy_store(db_file=DB_FILE):
    with sqlite3.connect(db_file) as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS policy_chunks (
                chunk_id TEXT PRIMARY KEY,
                section TEXT,
                chunk_text TEXT NOT NULL,
                embedding_json TEXT NOT NULL,
                updated_at TEXT NOT NULL DEFAULT CURRENT_TIMESTAMP
            )
        """)
        conn.commit()

def upsert_policy_text(policy_text: str = SAMPLE_HR_POLICY):
    """Chunk, embed, and persist policy chunks into SQLite."""
    init_policy_store()

    chunks = chunk_text(policy_text)
    with sqlite3.connect(DB_FILE) as conn:
        for chunk in chunks:
            vector = embed_text(chunk["text"])
            section = chunk["text"].split(":")[0][:80] if ":" in chunk["text"][:120] else "HR Policy"
            conn.execute(
                """
                INSERT INTO policy_chunks(chunk_id, section, chunk_text, embedding_json, updated_at)
                VALUES (?, ?, ?, ?, CURRENT_TIMESTAMP)
                ON CONFLICT(chunk_id) DO UPDATE SET
                    section = excluded.section,
                    chunk_text = excluded.chunk_text,
                    embedding_json = excluded.embedding_json,
                    updated_at = CURRENT_TIMESTAMP
                """,
                (chunk["chunk_id"], section, chunk["text"], json.dumps(vector)),
            )
        conn.commit()

    return len(chunks)

def query_hr_policy(question: str, top_k: int = 3):
    """Retrieve the most similar policy chunks from the SQLite vector-like store."""
    init_policy_store()
    query_vec = embed_text(question)

    with sqlite3.connect(DB_FILE) as conn:
        conn.row_factory = sqlite3.Row
        rows = conn.execute("SELECT chunk_id, section, chunk_text, embedding_json FROM policy_chunks").fetchall()

    scored = []
    for row in rows:
        vector = json.loads(row["embedding_json"])
        score = cosine_similarity(query_vec, vector)
        scored.append({
            "chunk_id": row["chunk_id"],
            "section": row["section"],
            "chunk_text": row["chunk_text"],
            "score": round(score, 4),
        })

    return sorted(scored, key=lambda x: x["score"], reverse=True)[:top_k]

num_chunks = upsert_policy_text()
print("Policy chunks stored:", num_chunks)
query_hr_policy("How many days in advance should annual leave be requested?")

Policy chunks stored: 3


[{'chunk_id': 'policy_chunk_001',
  'section': 'Annual Leave Policy',
  'chunk_text': 'Annual Leave Policy: Employees are entitled to annual leave according to their employment contract and leave register. Annual leave should normally be requested at least 5 working days in advance. Requests longer than 3 consecutive working days require reporting manager approval. Employees should not plan annual leave that exceeds their available annual leave balance. Sick Leave Policy: Employees may use sick leave for illness, medical appointments, or recovery. For sick leave longer than 2 consecutive working days, the employee may be asked to submit medical documentation. Sick leave is not',
  'score': 0.5397},
 {'chunk_id': 'policy_chunk_002',
  'section': 'HR Policy',
  'chunk_text': 'leave longer than 2 consecutive working days, the employee may be asked to submit medical documentation. Sick leave is not normally encashed. Casual Leave Policy: Casual leave can be used for urgent personal work or

In [ ]:
#Play with questions here:
query_hr_policy("How many days in advance should annual leave be requested?")

## Quick Test

Ask a policy question and inspect the retrieved chunks.

In [ ]:
results = query_hr_policy("What approval is required for more than 3 days of annual leave?")
pd.DataFrame(results)

,chunk_id,section,chunk_text,score
0,policy_chunk_001,Annual Leave Policy,Annual Leave Policy: Employees are entitled to...,0.5317
1,policy_chunk_002,HR Policy,"leave longer than 2 consecutive working days, ...",0.4250
2,policy_chunk_003,HR Policy,on a weekend or company holiday with prior app...,0.4064


# 3. Integrate Memory + Tools + RAG

The full HR Bot flow is:

```text
User message
  ↓
Save message to SQLite episodic memory
  ↓
Resolve employee ID:
  current message → short-term session memory → last 10 messages → ask user
  ↓
Retrieve relevant HR policy chunks
  ↓
Fetch employee and leave data if employee ID exists
  ↓
Build prompt with:
  summary + last 10 messages + employee context + policy chunks
  ↓
Generate answer
  ↓
Save assistant answer to SQLite episodic memory
  ↓
Compact older messages into short-term memory if more than 10 messages exist
```

In [ ]:
def build_hr_prompt_with_memory_and_rag(session_id: str, user_message: str, policy_results: list[dict], employee_context: dict | None):
    memory = get_session_memory(session_id)
    recent = get_recent_messages(session_id, limit=MAX_EPISODIC_MESSAGES)

    recent_text = "\n".join(f'{m["role"].upper()}: {m["content"]}' for m in recent) or "(no recent messages)"
    policy_text = "\n\n".join(
        f'[{r["chunk_id"]} | score={r["score"]}]\n{r["chunk_text"]}'
        for r in policy_results
    ) or "(no policy chunks retrieved)"

    return f"""
SYSTEM:
You are an HR Bot. Answer only from the employee tools, leave tools, policy retrieval, and conversation memory below.
If employee-specific information is needed and employee ID is missing, ask for it. Do not guess.

SHORT-TERM MEMORY SUMMARY:
{memory["summary"] or "(none)"}

CURRENT EMPLOYEE ID FOR THIS SESSION:
{memory["current_employee_id"] or "(not provided)"}

EPISODIC MEMORY — LAST {MAX_EPISODIC_MESSAGES} MESSAGES:
{recent_text}

EMPLOYEE / LEAVE TOOL CONTEXT:
{json.dumps(employee_context, indent=2) if employee_context else "(no employee-specific context)"}

RETRIEVED HR POLICY CONTEXT:
{policy_text}

USER MESSAGE:
{user_message}
""".strip()

def deterministic_hr_answer(user_message: str, emp_id: str | None, source: str, policy_results: list[dict]):
    """
    Deterministic fallback so the notebook runs without an LLM API key.
    In production, send build_hr_prompt_with_memory_and_rag(...) to Groq/OpenRouter or another approved LLM.
    """
    lower = user_message.lower()

    if any(word in lower for word in ["my", "balance", "leave days", "how many"]) and not emp_id:
        return (
            "I can help, but I do not know which employee record to check in this chat session. "
            "Please share your employee ID, for example EMP-1001."
        )

    if emp_id and any(word in lower for word in ["balance", "how many", "leave days"]):
        return answer_leave_balance(emp_id) + f" [Employee ID source: {source}]"

    if emp_id and any(word in lower for word in ["manager", "approve", "approval"]):
        manager_answer = answer_manager(emp_id)
        policy_hint = policy_results[0]["chunk_text"] if policy_results else ""
        return f"{manager_answer} Policy context: {policy_hint} [Employee ID source: {source}]"

    if policy_results:
        return "Relevant HR policy guidance: " + policy_results[0]["chunk_text"]

    return "I can help with HR policy, leave balance, manager approval, and employee-specific leave questions."

def hr_bot_with_rag(session_id: str, user_message: str):
    save_message(session_id, "user", user_message)

    emp_id, source = resolve_employee_id(session_id, user_message)
    policy_results = query_hr_policy(user_message, top_k=3)

    employee_context = None
    if emp_id:
        employee_context = {
            "employee": get_employee_details(emp_id),
            "leave_balance": get_leave_balance(emp_id),
            "employee_id_source": source,
        }

    prompt = build_hr_prompt_with_memory_and_rag(session_id, user_message, policy_results, employee_context)
    response = deterministic_hr_answer(user_message, emp_id, source, policy_results)

    save_message(session_id, "assistant", response)

    return {
        "response": response,
        "prompt_that_would_go_to_llm": prompt,
        "retrieved_policy_chunks": policy_results,
        "employee_id": emp_id,
        "employee_id_source": source,
    }

# 4. Test Scenario A — Employee ID Missing

Expected behavior:

- The user asks an employee-specific question.
- The current chat session does not contain employee ID.
- The bot asks for employee ID instead of guessing.

In [ ]:
session_id = "day2_missing_id"
reset_session(session_id)

result = hr_bot_with_rag(session_id, "How many annual leave days do I have?")
print(result["response"])

I can help, but I do not know which employee record to check in this chat session. Please share your employee ID, for example EMP-1001.


# 5. Test Scenario B — Employee ID Already Provided in This Session

Expected behavior:

- User provides employee ID once.
- Later question omits employee ID.
- Bot resolves employee ID from short-term session memory.

In [ ]:
session_id = "day2_known_id"
reset_session(session_id)

print(hr_bot_with_rag(session_id, "My employee ID is EMP-1004.")["response"])
print(hr_bot_with_rag(session_id, "How many annual leave days do I have?")["response"])
print(hr_bot_with_rag(session_id, "Who approves my leave?")["response"])

get_session_memory(session_id)

Relevant HR policy guidance: leave longer than 2 consecutive working days, the employee may be asked to submit medical documentation. Sick leave is not normally encashed. Casual Leave Policy: Casual leave can be used for urgent personal work or short unplanned absences. Casual leave should be requested as early as possible. Manager approval is required before the absence unless it is an emergency. Compensatory Off Policy: Comp-off may be granted when an employee works on a weekend or company holiday with prior approval. Comp-off must usually be used within the same leave year unless
Karan Bose (EMP-1004) has 14 annual leave days, 10 sick leave days, 5 casual leave days, and 2 comp-off days available for 2025-26. [Employee ID source: short_term_session_memory]
Karan Bose (EMP-1004) reports to Anita Desai. Policy context: Annual Leave Policy: Employees are entitled to annual leave according to their employment contract and leave register. Annual leave should normally be requested at leas

{'session_id': 'day2_known_id',
 'summary': '',
 'current_employee_id': 'EMP-1004',
 'updated_at': '2026-07-19 06:38:22'}

# 6. Inspect the Prompt Sent to the LLM

This cell shows the exact prompt package. Notice:

- the short-term memory summary,
- the current employee ID,
- the last 10 episodic messages only,
- employee and leave tool context,
- retrieved HR policy context.

In [ ]:
result = hr_bot_with_rag("day2_known_id", "Can I take more than 3 consecutive annual leave days?")
print(result["prompt_that_would_go_to_llm"])

SYSTEM:
You are an HR Bot. Answer only from the employee tools, leave tools, policy retrieval, and conversation memory below.
If employee-specific information is needed and employee ID is missing, ask for it. Do not guess.

SHORT-TERM MEMORY SUMMARY:
(none)

CURRENT EMPLOYEE ID FOR THIS SESSION:
EMP-1004

EPISODIC MEMORY — LAST 10 MESSAGES:
USER: My employee ID is EMP-1004.
ASSISTANT: Relevant HR policy guidance: leave longer than 2 consecutive working days, the employee may be asked to submit medical documentation. Sick leave is not normally encashed. Casual Leave Policy: Casual leave can be used for urgent personal work or short unplanned absences. Casual leave should be requested as early as possible. Manager approval is required before the absence unless it is an emergency. Compensatory Off Policy: Comp-off may be granted when an employee works on a weekend or company holiday with prior approval. Comp-off must usually be used within the same leave year unless
USER: How many annual 

# 7. Trigger Last-10 Compaction Again

We will add enough messages to prove that the complete bot still enforces the same memory rule.

In [ ]:
session_id = "day2_compaction_test"
reset_session(session_id)

for i in range(1, 9):
    hr_bot_with_rag(session_id, f"My employee ID is EMP-1001. Question {i}: What is the annual leave rule?")

recent = get_recent_messages(session_id)
memory = get_session_memory(session_id)

print("Messages kept in episodic memory:", len(recent))
print("\nCompressed short-term summary:\n")
print(memory["summary"])

pd.DataFrame(recent)

Messages kept in episodic memory: 10

Compressed short-term summary:

Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'annual']. Recent older context: user: My employee ID is EMP-1001. Question 1: What is the annual leave rule?
Compressed 1 older messages. Employee IDs mentioned: none. Topics: ['leave', 'manager', 'policy', 'sick', 'casual', 'approval']. Recent older context: assistant: Relevant HR policy guidance: leave longer than 2 consecutive working days, the employee ma
Compressed 1 older messages. Employee IDs mentioned: ['EMP-1001']. Topics: ['leave', 'annual']. Recent older context: user: My employee ID is EMP-1001. Question 2: What is the annual leave rule?
Compressed 1 older messages. Employee IDs mentioned: none. Topics: ['leave', 'manager', 'policy', 'sick', 'casual', 'approval']. Recent older context: assistant: Relevant HR policy guidance: leave longer than 2 consecutive working days, the employee ma
Compressed 1 older messages. Emplo

,id,role,content,created_at
0,17,user,My employee ID is EMP-1001. Question 4: What i...,2026-07-13 16:07:59
1,18,assistant,Relevant HR policy guidance: leave longer than...,2026-07-13 16:07:59
2,19,user,My employee ID is EMP-1001. Question 5: What i...,2026-07-13 16:07:59
3,20,assistant,Relevant HR policy guidance: leave longer than...,2026-07-13 16:07:59
4,21,user,My employee ID is EMP-1001. Question 6: What i...,2026-07-13 16:07:59
5,22,assistant,Relevant HR policy guidance: leave longer than...,2026-07-13 16:07:59
6,23,user,My employee ID is EMP-1001. Question 7: What i...,2026-07-13 16:08:00
7,24,assistant,Relevant HR policy guidance: leave longer than...,2026-07-13 16:08:00
8,25,user,My employee ID is EMP-1001. Question 8: What i...,2026-07-13 16:08:00
9,26,assistant,Relevant HR policy guidance: leave longer than...,2026-07-13 16:08:00


# 8. Practice Extensions

Choose one:

1. Replace the local `embed_text()` function with a hosted embedding model.
2. Replace the SQLite policy store with Pinecone, Qdrant, Weaviate, or pgvector.
3. Add a policy source column and retrieve only from “leave policy”.
4. Add a permission check before showing salary or performance information.
5. Add a real LLM call using the generated prompt.

# Day 2 Wrap-Up

You now have the full HR Bot architecture:

```text
Episodic memory: last 10 chat messages in SQLite
Short-term memory: compressed session summary in SQLite
Operational tools: employee and leave SQLite queries
Long-term policy memory: policy chunks and vectors in SQLite
Prompt assembly: summary + recent messages + tool context + retrieved chunks
```

The same design can later be upgraded:

- SQLite episodic memory → Redis
- SQLite short-term memory → PostgreSQL
- SQLite policy vector store → Pinecone or another vector DB
- deterministic answer fallback → hosted LLM generation

## Optional free-provider upgrade

Reuse the provider helper from Week 5 to send the assembled HR prompt to Groq or OpenRouter. Keep employee records and sensitive HR data anonymised in classroom exercises, and do not send real personal data to any hosted model without organisational approval.